# HSCIS-ESD Analysis
Hybrid Symbolic Clinical Inference System â€” full evaluation notebook

In [1]:
import warnings
warnings.filterwarnings('ignore')
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np

from src.data.loader import load_dataset
from src.grading.fuzzy_grader import FuzzyGrader
from src.grading.feature_engineer import FeatureEngineer
from src.symbolic.pipeline import SymbolicPipeline
from src.models.model_a import run_model_a
from src.models.model_b import run_model_b
from src.models.model_c import run_model_c
from src.models.model_ensemble import run_model_ensemble
from src.triage.biopsy_triage import BiopsyTriage
from src.evaluation.metrics import print_comparison_table, run_mcnemar_test, per_class_safety_analysis
from src.evaluation.explainability import train_final_model, compute_shap_values, plot_shap_global, plot_shap_beeswarm, extract_imodels_rules

X_clinical, X_histopath, X_all, y = load_dataset()
print(f'Dataset: {X_all.shape}, Classes: {y.value_counts().to_dict()}')

Dataset: (366, 34), Classes: {0: 112, 2: 72, 1: 61, 4: 52, 3: 49, 5: 20}


## Model Comparison (A vs B vs C)

In [2]:
res_a = run_model_a(X_all, y)
res_b = run_model_b(X_clinical, y)
res_c = run_model_c(X_clinical, y)
res_d = run_model_ensemble(X_clinical, y)
print_comparison_table(res_a, res_b, res_c, res_d)


Model                                                  Accuracy         Macro F1
----------------------------------------------------------------------------------
Model A (All 34 features)                      0.9671 +/- 0.0166 0.9630 +/- 0.0180
Model B (12 clinical features)                 0.8495 +/- 0.0601 0.8424 +/- 0.0590
Model C (12 fuzzy + 8 engineered + 9 symbolic) 0.8634 +/- 0.0456 0.8551 +/- 0.0570
Model D (Ensemble: XGB + RF + LGBM, 29 features) 0.8688 +/- 0.0272 0.8624 +/- 0.0318

Per-class F1 scores:
Disease                                  Mdl A      Mdl B      Mdl C      Mdl D
-------------------------------------------------------------------------------
psoriasis                               1.0000     0.9364     0.9286     0.9375
seborrheic_dermatitis                   0.9091     0.6667     0.7344     0.6984
lichen_planus                           0.9930     0.9859     0.9640     0.9930
pityriasis_rosea                        0.8889     0.7619     0.7835     0.8163

## Statistical Test: B vs C

In [3]:
r_bc = run_mcnemar_test(res_b, res_c)
print("McNemar Test (B vs C):")
print(f"  a={r_bc['a']}  b={r_bc['b']}  c={r_bc['c']}  d={r_bc['d']}")
print(f"  chi2={r_bc['chi2_stat']}  p={r_bc['p_value']}  significant={r_bc['significant']}")
print(f"  {r_bc['interpretation']}")
print()
r_cd = run_mcnemar_test(res_c, res_d)
print("McNemar Test (C vs D/Ensemble):")
print(f"  a={r_cd['a']}  b(C wins)={r_cd['b']}  c(D wins)={r_cd['c']}  d={r_cd['d']}")
print(f"  chi2={r_cd['chi2_stat']}  p={r_cd['p_value']}  significant={r_cd['significant']}")
print(f"  {r_cd['interpretation']}")

McNemar Test (B vs C):
  a=300  b=11  c=16  d=39
  chi2=0.5926  p=0.4414  significant=False
  No significant difference (p=0.4414): C correct on 16 cases B missed; B correct on 11 cases C missed.

McNemar Test (C vs D/Ensemble):
  a=307  b(C wins)=9  c(D wins)=11  d=39
  chi2=0.05  p=0.8231  significant=False
  No significant difference (p=0.8231): C correct on 11 cases B missed; B correct on 9 cases C missed.


## SHAP Analysis â€” Model C

In [4]:
X_combined = res_c['X_combined']
final_model = train_final_model(X_combined, y)
shap_values = compute_shap_values(final_model, X_combined)
plot_shap_global(shap_values, X_combined, save_path='shap_global.png')
plot_shap_beeswarm(shap_values, X_combined, save_path='shap_psoriasis.png', class_idx=0)
print('SHAP plots saved.')

Saved: shap_global.png
Saved: shap_psoriasis.png
SHAP plots saved.


## imodels Rule Extraction

In [5]:
rules = extract_imodels_rules(X_combined, y, max_rules=15)
print('Top IF-THEN rules (post-hoc, from Model C):')
for r in rules:
    print(' ', r)

Top IF-THEN rules (post-hoc, from Model C):
  imodels rule extraction failed: RuleFit does not yet support multiclass classification


## Biopsy Triage + Per-Class Safety Analysis

In [8]:
grader = FuzzyGrader()
pipeline = SymbolicPipeline("../rules")
X_fuzzy = grader.grade(X_clinical).reset_index(drop=True)
X_symbolic = pipeline.transform(X_fuzzy)

final_model_c = train_final_model(res_c["X_combined"], y)
y_pred = final_model_c.predict(res_c["X_combined"])

safety_df = per_class_safety_analysis(X_symbolic, y, y_pred)
print(safety_df.to_string(index=False))

                 disease  n_patients  n_correct  n_safe_biopsy_free  pct_safe_biopsy_free
               psoriasis         112        112                   8                   7.1
   seborrheic_dermatitis          61         61                   3                   4.9
           lichen_planus          72         72                   6                   8.3
        pityriasis_rosea          49         49                   0                   0.0
      chronic_dermatitis          52         51                   0                   0.0
pityriasis_rubra_pilaris          20         20                   0                   0.0


## Single Patient Reasoning Trace

In [9]:
sample = X_clinical.iloc[0]
fuzzy_sample = grader.grade_series(sample)
trace = pipeline.explain(fuzzy_sample)

print("=== Clinical Reasoning Trace ===")
print("Certainty scores:", trace["certainty_scores"])
print("Conflict load:   ", trace["conflict_load"])
print("Contradiction:   ", trace["contradiction_severity"])
print("FSM state:       ", trace["fsm_state"])
print("Fired rules:")
for r in trace["fired_rules"]:
    print(f"  {r['id']} ({r['disease']}, tier {r['tier']}): strength={r['firing_strength']}, contrib={r['contribution']}")

triage = BiopsyTriage()
certainty_vals = list(trace["certainty_scores"].values())
triage_result = triage.recommend(
    top_certainty=max(certainty_vals),
    conflict_load=trace["conflict_load"],
    fsm_state=trace["fsm_state"],
)
print()
print("Triage recommendation:", triage_result)
print("True label:", y.iloc[0])


SyntaxError: unterminated f-string literal (detected at line 21) (4104046556.py, line 21)